### Application du lisseur pour le traitement de données

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

In [ ]:
folder_name = 'Tunis_F2/Day_2/src/'
root_dir = '/content/gdrive/My Drive/'
base_dir = root_dir + folder_name

In [ ]:
pip install scienceplots

In [ ]:
import numpy as np
import pandas as pd
import os
import sys
import datetime

current_dir = os.getcwd()
module_path = os.path.abspath(os.path.join(base_dir, '..', 'libs'))
if module_path not in sys.path:
    sys.path.append(module_path)

from SCOU_extended_3_1 import *
from utils import *

In [ ]:
data = pd.read_csv(base_dir.split('src')[0] + '/data/paris_dataset.csv', sep=";")

In [ ]:
data

# Traitement de PARIS_MARNE_AVAL

### Etape 1 : Récupération des lignes qui correspondent à cette station

In [ ]:
# Definition du nom de la station
station = 'PARIS_MARNE_AVAL'

# Extraction des lignes correspondant à cette station
sub_data = data.loc[data.plantName==station].copy()

# Suppression de la colonne 'plantName' qui n'a plus d'utilisé car il n'y a qu'une occurence:
sub_data = sub_data.drop('plantName', axis=1)

# Affichage des 5 premières lignes:
sub_data.head()

### Etape 2 : Traitement des effets de dilution

In [ ]:
# Conversion des volumes de m3 en L :
for col in ['plantVolume', 'meanVolume']:
    sub_data[col] *= 1000

# Passage en flux, puis division par le volume moyen si on souhaite comparer plusieurs stations entre-elles :
#for col in ['obs', 'lod']:
#    sub_data[col] = sub_data[col] * sub_data['plantVolume'] / sub_data['meanVolume']

# Mais ici on ne traite qu'une station. On peut donc se contenter de travailler en flux !
for col in ['obs', 'lod']:
    sub_data[col] = sub_data[col] * sub_data['plantVolume']

sub_data.head()

### Etape 3 : Rajout des jours manquants au calendrier

In [ ]:
# Conversion de la colonne dateStart au format datetime:
sub_data.dateStart = pd.to_datetime(sub_data.dateStart)

# Rajout des jours au calendrier en une seule ligne de code:
sub_data = sub_data.set_index('dateStart').resample('D').mean().reset_index()

sub_data.head()

### Etape 4 : Passage en échelle log pour être conforme avec les hypothèses de l'algorithme de lissage

In [ ]:
# Il ne faut pas oublier d'effectuer cette transformation pour la colonne obs ET pour la colonne lod
for col in ['obs', 'lod']:
    sub_data[col] = np.log10(sub_data[col])

sub_data.head()

### Etape 5 : Utilisation des fonctions prédéfinies pour le lissage

In [ ]:
scou = model_fit(sub_data, 'obs', 'lod')

In [ ]:
get_results(sub_data, scou, remove_those = [])